# AML Kafka Producer

Lê `LI-Medium_Trans.csv` de HDFS, serializa cada linha em JSON e envia para o tópico Kafka `transactions`
em rajadas controladas (simulando a chegada de dados em streaming de `x` em `x` segundos).

Usa o sink nativo Kafka do Spark (`spark.write.format("kafka")`), pelo que distribui a escrita pelo cluster.

**Pré-requisito:** o connector `spark-sql-kafka-0-10` tem de estar disponível no classpath do cluster
(habitualmente já instalado em `$SPARK_HOME/jars/`).

In [ ]:
import time
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField,
    TimestampType, IntegerType, StringType, DoubleType,
)

KAFKA_BOOTSTRAP = "10.84.129.52:9092"
KAFKA_TOPIC     = "transactions"

spark = SparkSession.builder \
    .appName("AML Kafka Producer") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(f"Kafka bootstrap: {KAFKA_BOOTSTRAP}")
print(f"Topic: {KAFKA_TOPIC}")

In [ ]:
HDFS_BASE = "hdfs://10.84.129.52:9000/aulas/fabricio_moreira/project"
SOURCE_PATH = f"{HDFS_BASE}/dataset/LI-Medium_Trans.csv"

# Parâmetros do producer (afináveis na demo)
NUM_CHUNKS    = 20       # número de rajadas
DELAY_SECONDS = 5        # intervalo entre rajadas
MAX_RECORDS   = 200_000  # 0 ou None = enviar tudo (atenção ao tamanho do Medium ~30M linhas)

aml_schema = StructType([
    StructField("Timestamp", TimestampType(), True),
    StructField("From Bank", IntegerType(), True),
    StructField("Account2", StringType(), True),
    StructField("To Bank", IntegerType(), True),
    StructField("Account4", StringType(), True),
    StructField("Amount Received", DoubleType(), True),
    StructField("Receiving Currency", StringType(), True),
    StructField("Amount Paid", DoubleType(), True),
    StructField("Payment Currency", StringType(), True),
    StructField("Payment Format", StringType(), True),
    StructField("Is Laundering", IntegerType(), True),
])

In [ ]:
df = spark.read \
    .option("header", True) \
    .option("timestampFormat", "yyyy/MM/dd HH:mm") \
    .schema(aml_schema) \
    .csv(SOURCE_PATH)

if MAX_RECORDS:
    df = df.limit(MAX_RECORDS)

# `value` é a coluna obrigatória do sink Kafka (JSON serializado em string)
df_kafka = df.select(
    F.to_json(F.struct([F.col(c).alias(c) for c in df.columns])).alias("value")
)
df_kafka.cache()
total = df_kafka.count()
print(f"Total de mensagens a enviar: {total:,} em {NUM_CHUNKS} rajadas de ~{total // NUM_CHUNKS:,} msgs")

In [ ]:
# Divide aleatoriamente em chunks para simular rajadas no tempo
chunks = df_kafka.randomSplit([1.0] * NUM_CHUNKS, seed=42)

t0 = time.time()
sent_total = 0

for i, chunk in enumerate(chunks, start=1):
    t_chunk = time.time()
    chunk.write \
        .format("kafka") \
        .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP) \
        .option("topic", KAFKA_TOPIC) \
        .save()
    n = chunk.count()  # já em cache via parent
    sent_total += n
    dt = time.time() - t_chunk
    print(f"Rajada {i:02d}/{NUM_CHUNKS}: {n:>7,} msgs em {dt:5.1f}s  (cumul: {sent_total:,})")
    if i < NUM_CHUNKS:
        time.sleep(DELAY_SECONDS)

elapsed = time.time() - t0
print(f"\nTotal: {sent_total:,} msgs em {elapsed:.1f}s ({sent_total/elapsed:,.0f} msgs/s)")